# Problem 22: Tool Fallback Strategy

**Difficulty:** Medium | **Framework:** LangChain

**Categories:** tool-calling, error-recovery

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/tool-fallback-strategy).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The primary search tool must be tried first
- Fallback to the secondary search only when the primary fails or returns no results
- The agent must not call both tools simultaneously on every query
- Error messages from the primary tool must be caught, not surfaced raw to the user


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

llm = ChatOpenAI(model="gpt-4o-mini")

@tool
def primary_search(query: str) -> str:
    """Search using the primary search engine (sometimes fails)."""
    # BUG: This tool intermittently fails and there is no fallback
    import random
    if random.random() < 0.5:
        raise ConnectionError("Primary search service is temporarily unavailable")
    return f"Primary results for '{query}': Article about {query} from TechNews, Research paper on {query} from ArXiv"

@tool
def backup_search(query: str) -> str:
    """Search using the backup search engine (always works)."""
    return f"Backup results for '{query}': Blog post about {query}, Wikipedia article on {query}"

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant. Search for information to answer questions."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# BUG: Only primary_search is registered — when it fails, the agent crashes
# TODO: Add fallback logic so backup_search is used when primary fails
agent = create_tool_calling_agent(llm, [primary_search], prompt)
executor = AgentExecutor(agent=agent, tools=[primary_search])

result = executor.invoke({"input": "Find recent research on transformer architectures"})
print(result["output"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Wrap the primary tool call in error handling that catches exceptions and triggers the fallback
2. In LangGraph, model this as a conditional edge: if primary search fails, route to the backup search node
3. Consider having the primary tool return a sentinel value on failure so the agent knows to try the fallback


## 7. Evaluation Criteria

Check your solution against these criteria:

- Primary search is tried first
- Fallback triggers on failure
- Agent never crashes
- No unnecessary double calls
